# 03 · Reconcile

**Don't just calculate Δ. Explain Δ.**

1. How much do the scores disagree, and is the disagreement *structured*?
2. For every flagged member, *why*?
3. What should be investigated first?

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.pipeline import run
res = run(write=False)
c = res['classified']
res['metrics']['headline']

{'n_members': 25000,
 'pct_beyond_tolerance': 0.06908,
 'n_beyond_tolerance': 1727,
 'median_delta': 0.0040000000000000036,
 'mae': 0.21687523999999997,
 'p95_abs_delta': 0.6499999999999999}

## Is disagreement random — or structured?

In [2]:
from src.reconciliation import completeness_lift
for name, t in res['segments'].items():
    print(name); print(t.round(3).to_string(), '\n')
print(completeness_lift(c))

completeness_band
                       n  mean_abs_delta  median_delta  pct_beyond_tolerance  lift_vs_overall
completeness_band                                                                            
<0.70               1226           0.361        -0.205                 0.257            3.719
0.70-0.80            434           0.261        -0.093                 0.136            1.968
0.80-0.90           2118           0.214         0.015                 0.054            0.786
>=0.90             21222           0.208         0.015                 0.058            0.844 

utilization_band
                      n  mean_abs_delta  median_delta  pct_beyond_tolerance  lift_vs_overall
utilization_band                                                                            
none               2372           0.245        -0.199                 0.124            1.794
low               12165           0.168        -0.026                 0.056            0.807
medium             7601    

## Why do they disagree?
Each flagged member is tested against four defect mechanisms. A defect is named only if its diagnostic fires **and** it explains enough of Δ in the right direction. Otherwise the gap is compared with clean peers of similar utilization: ordinary variance, or `UNEXPLAINED`.

In [3]:
res['causes'].round(3)

,cause,flagged_members,mean_abs_delta,mean_priority,total_priority,share_of_flagged,tier
0,DUPLICATE_INPUT,70,0.775,0.210,14.727,0.041,Critical
1,DATA_COMPLETENESS,254,0.727,0.198,50.363,0.147,High
2,SNAPSHOT_MISMATCH,572,1.267,0.465,265.961,0.331,High
3,MODEL_VERSION,243,0.648,0.179,43.476,0.141,High
4,UNEXPLAINED,232,1.569,0.418,96.871,0.134,Investigate
5,EXPECTED_MODEL_VARIANCE,356,0.503,0.023,8.325,0.206,Monitor


## Does the framework recover what I planted?
Rows = planted primary cause, columns = what the framework said (flagged members only).

In [4]:
print(res['recovery_confusion'].to_string())
print()
print(res['recovery_metrics'].round(3).to_string(index=False))
print('\nOverall agreement:', round(res['metrics']['recovery_accuracy'], 3))

cause                    DATA_COMPLETENESS  SNAPSHOT_MISMATCH  DUPLICATE_INPUT  MODEL_VERSION  EXPECTED_MODEL_VARIANCE  UNEXPLAINED
planted                                                                                                                            
DATA_COMPLETENESS                      205                  2                0              0                        3           56
SNAPSHOT_MISMATCH                        0                548                0              0                        0            0
DUPLICATE_INPUT                          0                  1               68              0                        0            0
MODEL_VERSION                            0                  0                0            221                        0            0
EXPECTED_MODEL_VARIANCE                 49                 21                2             22                      351           43
UNEXPLAINED                              0                  0               

**Where it falls short (on purpose, visibly):** the misses fall into two groups. Some `DATA_COMPLETENESS` cases land in `UNEXPLAINED` because the unseen claims happened to be the expensive ones, which no member-level diagnostic can reveal. And some ordinary `METHOD` gaps coincide with a weak diagnostic (e.g. slightly low completeness) and get credited to it, while a few heavy-tail ones cross the robust-z line. This grades the framework on data I generated alongside a framework I designed, so it tests internal consistency, not real-world performance.

## The data product: an investigation queue
Not 1,727 mismatched records — the cases to look at first, and why.

In [5]:
q = res['queue']
q.head(15)[['rank','member_id','ds_risk_score','actuarial_risk_score','delta','cause','evidence','priority']].round(3)

,rank,member_id,ds_risk_score,actuarial_risk_score,delta,cause,evidence,priority
0,1,M08554,4.128,7.893,-3.765,SNAPSHOT_MISMATCH,snapshot_gap_days=73; gap_claim_load=3.86; DS_...,0.936
1,2,M12308,2.548,5.472,-2.924,SNAPSHOT_MISMATCH,snapshot_gap_days=61; gap_claim_load=2.94; DS_...,0.927
2,3,M24840,1.255,4.551,-3.296,SNAPSHOT_MISMATCH,snapshot_gap_days=81; gap_claim_load=2.86; DS_...,0.922
3,4,M10964,2.340,5.795,-3.455,SNAPSHOT_MISMATCH,snapshot_gap_days=42; gap_claim_load=2.86; DS_...,0.921
4,5,M18339,2.798,5.908,-3.110,SNAPSHOT_MISMATCH,snapshot_gap_days=36; gap_claim_load=2.86; DS_...,0.909
5,6,M01349,4.885,8.772,-3.887,UNEXPLAINED,robust_z=-13.0; no_diagnostic_gate_explains_delta,0.903
6,7,M17004,0.877,4.020,-3.143,SNAPSHOT_MISMATCH,snapshot_gap_days=39; gap_claim_load=2.36; DS_...,0.902
7,8,M10348,1.343,4.089,-2.746,SNAPSHOT_MISMATCH,snapshot_gap_days=64; gap_claim_load=2.86; DS_...,0.902
8,9,M12088,1.255,4.060,-2.805,SNAPSHOT_MISMATCH,snapshot_gap_days=82; gap_claim_load=2.86; DS_...,0.899
9,10,M10667,1.558,4.423,-2.865,SNAPSHOT_MISMATCH,snapshot_gap_days=46; gap_claim_load=2.78; DS_...,0.898


In [6]:
q.cause.value_counts()

cause
SNAPSHOT_MISMATCH    45
UNEXPLAINED           5
Name: count, dtype: int64

The queue is dominated by stale-snapshot cases because they have the largest, most decision-relevant gaps. `DUPLICATE_INPUT` is tiered **Critical** at the *cause* level instead — one upstream fix clears every affected member — which is why the cause table and the case queue answer different questions.

## Optional: Reconciliation Copilot guardrail
The LLM would see only deterministic diagnostics, and its explanation is rejected if it cites evidence that isn't in the packet. Here is the no-LLM baseline and the guardrail in action (no API call is made in this notebook).

In [7]:
from src.copilot import evidence_packet, deterministic_summary, validate_explanation
row = c[c.cause=='DATA_COMPLETENESS'].sort_values('priority', ascending=False).iloc[0]
packet = evidence_packet(row)
print(packet)
expl = deterministic_summary(packet); print(expl)
print('valid:', validate_explanation(expl, packet) == [])
invented = dict(expl, evidence=['claims_completeness=0.30'])
print('invented evidence ->', validate_explanation(invented, packet))

{'member_id': 'M17415', 'ds_risk_score': '3.86', 'actuarial_risk_score': '5.76', 'delta': '-1.90', 'claims_completeness': '0.47', 'snapshot_gap_days': '0', 'gap_claim_load': '0.00', 'dup_claim_ratio': '0.00', 'ds_score_version': 'v2.4', 'rx_count': '8', 'robust_z': '-11.06', 'explained_fraction': '0.31', 'unexplained_residual': '1.30', 'cause': 'DATA_COMPLETENESS'}
{'category': 'DATA_COMPLETENESS', 'evidence': ['claims_completeness=0.47'], 'next_step': 'Re-run the DS score after claims run-out completes; compare against the completed extract.'}
valid: True
invented evidence -> ["evidence 'claims_completeness=0.30' does not match the diagnostics"]
